# Week 8: Introduction to AI Agents & MCP

## 🎯 Session Goal
**Build your first AI Agent Tool.**

We are entering the era of "Agentic AI". It's no longer just about chatting with LLMs; it's about **giving them hands** to do things.

**The Standard: Model Context Protocol (MCP)**
Just like USB lets you connect any device to any computer, MCP lets you connect any data source/tool to any AI Agent (Claude, Cursor, etc.).

## 📋 The Plan
1.  **Understand**: What is a Server vs Client in AI?
2.  **Build**: A simple "Math & Time" Server (`simple_server.py`).
3.  **Connect**: A Python Client script (`simple_client.py`).

## Step 1: Install Dependencies
We need the `mcp` package.

In [ ]:
pip install -r requirements.txt

In [ ]:
s# TODO: Install the mcp package
# Hint: Use %pip install ___


## Step 2: The Server Code (`simple_server.py`)

We will write a python script that acts as an **MCP Server**.
It will expose three "Tools":
1.  `add_numbers`: Adds two integers.
2.  `multiply_numbers`: Multiplies two integers.
3.  `get_current_time`: Returns the current time.

**Note**: We use `%%writefile` to save this directly to a file.

sys.stderr:

stdout - Standard Output   - Normal Output
stderr - Standard Error    - Logs, debugging, errors
stdin - Standard Input     - Input

When I use
print() -> goes to stdout

if I write sys.stderr -> Its not ur actual output  this just for your logging purpose/ debugging purpose 

- MCP Communications work via structures messages on stdout
1. If I mix my logs or print statements it breaks the MCP protocol, it confuses the client and corrupt the responses
2. Tool responses go via MCP protocol
3. Logs directly in stderr 

In [14]:
%%writefile simple_server.py
# TODO: Import required libraries
# Hint: from mcp.server.fastmcp import ___
# Hint: import datetime, sys, json
from mcp.server.fastmcp import FastMCP
import datetime
import sys
import json


# TODO: Create the MCP Server
# Hint: mcp = FastMCP("___")
mcp = FastMCP("Math & Time Helper") # This creates your MCP server instance


# TODO: Define Tool 1 - add_numbers
# Hint: Use @mcp.tool() decorator
# Hint: def add_numbers(a: int, b: int) -> int:
@mcp.tool()   ## Registering this function as a callable tool for AI
def add_numbers(a: int, b: int) -> int:
    """Add two numbers togeather."""   # Document Strings are the languages between your tool and AI brain
    result = a + b
    print(f"add_numbers {a}, {b} = {result}", file = sys.stderr)
    return result



# TODO: Define Tool 2 - multiply_numbers
# Hint: Similar to add_numbers
@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers togeather"""
    result = a * b
    print(f"multiply numbers {a} , {b} = {result}", file = sys.stderr)
    return result



# TODO: Define Tool 3 - get_current_time
# Hint: No parameters needed, returns str
# Hint: Use datetime.datetime.now().isoformat()
@mcp.tool()
def get_current_time() -> str:
    """Get the current local time in ISO format"""
    result = datetime.datetime.now().isoformat() # This converts the datetime object into a string in ISO 86** format
    print(f"get_current_time() = {result}", file = sys.stderr)
    return result

# TODO: Run the server
# Hint: if __name__ == "__main__":
# Hint:     mcp.run()

if __name__ == "__main__":   ## The server runs only when this file is executed directly
    print("Starting the Math & Time Helper Server...", file = sys.stderr)
    print("Waiting for Client Connections...", file = sys.stderr)

    mcp.run()   ## Starts listening for MCP client connections, waits for AI tools to be called, responds to the requests

Overwriting simple_server.py


In [4]:
import datetime

datetime.datetime.now().isoformat()

'2026-01-25T12:11:25.984596'

## Step 3: The Client Code (`simple_client.py`)

Now we need something to talk to our server. Usually, this would be **Claude Desktop** or **Cursor**.
But to understand *how* it works, we will write a tiny Python script that acts as the Agent.

**Note on Complexity:** This code uses `async`/`await` because MCP requires asynchronous communication. Don't worry if you don't fully understand it yet - focus on the MCP concepts (Server, Tools, Discovery).

ClientSession - Like a phone call between your server-client
StdioServerParameters = How shoulf I start the server

Sync calls -> Tasks happen one after another
Each task must must finish before the next one starts

Async Calls:
Concurrent
not blocking
Non Blocking, uses async and await , suitable for API calls, Web Scrapping, Chatbots


In [15]:
%%writefile simple_client.py
# TODO: Import required libraries
# Hint: import asyncio, sys, os
# Hint: from mcp import ClientSession, StdioServerParameters
# Hint: from mcp.client.stdio import stdio_client
import asyncio # async code
from mcp import ClientSession, StdioServerParameters 
from mcp.client.stdio import stdio_client # Communication channels using stdin/ stdout
import sys
import os

# TODO: Step 1 - Find the server script
# Hint: server_script = os.path.join(os.path.dirname(__file__), "___")
server_script = os.path.join(os.path.dirname(__file__), "simple_server.py")

# TODO: Step 2 - Configure how to start the server
# Hint: server_params = StdioServerParameters(
#           command=sys.executable,
#           args=[___],
#       )
server_params = StdioServerParameters(
    command = sys.executable,  # Path to the interpreter
    args = [server_script],    # Run python simple_server.py
)

# TODO: Step 3 - Define the async client function
# Hint: async def run_client():
async def run_client():   #async - Server communication takes time, we dont want balls to be blocked
    print(f"Using Server Script: {server_script}")

    # TODO: Connect to server using stdio_client
    # Hint: async with stdio_client(___) as (read, write):
    try:
        async with stdio_client(server_params) as (read, write): # starts the server process, read -> received the messages from server | write -> Sens the messages to server
            print("Server Process Started")
        
            # TODO: Create a session
            # Hint: async with ClientSession(read, write) as session:
            async with ClientSession(read, write) as session:
                print("Session Created")
            
                # TODO: Initialize the session
                # Hint: await session.initialize()
                print("Initializing Session")
                await session.initialize()
                print("Connected")
                
                # TODO: List available tools
                # Hint: tools = await session.list_tools()
                tools = await session.list_tools()
                print(f"Available Tools: {[t.name for t in tools.tools]}")
                
                # TODO: Call add_numbers tool
                # Hint: result = await session.call_tool("___", arguments={"a": 10, "b": 20})
                print("Addition tool check")
                result = await session.call_tool("add_numbers", arguments={"a":10, "b": 20})
                print(f"Result: {result.content[0].text}")
                
                # TODO: Call multiply_numbers tool
                print("Multiply tool check")
                result = await session.call_tool("multiply_numbers", arguments={"a":5, "b": 6})
                print(f"Result: {result.content[0].text}")
                
                # TODO: Call get_current_time tool
                print("Testing: get current time tool")
                result = await session.call_tool("get_current_time", arguments={})
                print(f"Result: {result.content[0].text}")

                print("All the test passed")

    except Exception as e:
        print(f"Error: {e}")
        raise


# TODO: Step 4 - Run the async function
# Hint: if __name__ == "__main__":
# Hint:     asyncio.run(___)
if __name__ == "__main__":
    asyncio.run(run_client())

Overwriting simple_client.py


## Step 4: Run It!

**You only need ONE terminal!** The client automatically starts the server.

Open a **Terminal** and run:
```bash
python simple_client.py
```

You should see the Client connect to the Server, discover the tools, and get the answers!

## 🎓 What You Just Learned

1. **MCP Architecture**: Server (tools) ↔ Client (AI agent)
2. **Building Servers**: Use `FastMCP` and `@mcp.tool()` decorator
3. **Tool Design**: Clear docstrings + type hints
4. **Protocol**: stdio communication (stdin/stdout)
5. **Lifecycle Management**: Client controls server startup/shutdown

---

## 💡 Try it yourself!

**Exercise**: Add a new tool to the server
- Create a `subtract_numbers` function & `divide_numbers` function
- Update the server code
- Test it in the client

**Challenge**: Create a tool that:
- Takes a number and returns its square
- Takes a string and returns it reversed